# B01 — Audit header, metadata lengkap, dan split CBIS-DDSM

Gunakan **CPU**. Notebook ini melanjutkan laporan B00: membaca semua header DICOM CMMD/INbreast tanpa mendekode piksel, membaca CSV INbreast dengan `sep=";"`, serta mengekstrak PDF MIAS menggunakan pypdf. B00 sebelumnya salah membaca PDF sebagai teks biner; teks tersebut tidak digunakan.

**Hasil yang langsung dibuat:** pembagian baru CBIS-DDSM 70/15/15 berdasarkan pasien, seed split 2026, dari `cbis_all.csv`. Split lama dan checkpoint lama tidak diubah. **Hasil dataset lain:** audit pemetaan sisi/label dan dokumen asli; tidak menebak label atau identitas pasien. Tidak ada training atau konversi piksel dalam notebook ini.

Persetujuan pembimbing atas sumber Kaggle sudah dicatat dalam konteks proyek. Pemeriksaan berikut bertujuan memastikan ketepatan sampel/label, bukan meminta persetujuan sumber kembali.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                      'pydicom>=3,<4', 'pypdf>=5,<7', 'openpyxl', 'xlrd', 'scikit-learn'])
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib, json, io, zipfile, re, shutil, os, platform
import numpy as np
import pandas as pd
import pydicom
from pypdf import PdfReader
from sklearn.model_selection import train_test_split
from IPython.display import display

PROJECT = Path('/content/drive/MyDrive/Research')
ARCHIVES = {
    'cmmd': PROJECT/'datasets/TheChineseMammographyDatabase.zip',
    'inbreast': PROJECT/'datasets/inbreast_2012.zip',
    'mias': PROJECT/'datasets/miasdbv1.21.zip',
}
CBIS_CSV = PROJECT/'processed/cbis_all.csv'
EXPECTED_CBIS_SHA256 = 'a9eb6c7185b154021158fe06928bb44478953ff3e7019181cfccbf770d9c5139'
EXPECTED_INDEX = {
    'cmmd': '4046d320c36c5ac55fb0239d6289c573116d23ba8e4d952c6c5f05152b23be3f',
}
# Copy compressed archives once to the runtime for efficient header access.
COPY_ARCHIVES_LOCAL = True
LOCAL_ARCHIVES = Path('/content/benchmark_archive_cache')
BASE = PROJECT/'outputs/benchmark_4datasets'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = BASE/'preparation'/RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
CACHE = BASE/'header_cache_v1'
CACHE.mkdir(parents=True, exist_ok=True)
SPLIT_SEED = 2026
# Optional exact paths when automatic CBIS image lookup is incomplete.
CBIS_IMAGE_ROOTS = [PROJECT/'datasets', PROJECT, Path('/content'),
                    Path('/content/drive/MyDrive/datasets')]
CBIS_ZIP_OVERRIDE = ''
report = {'run_id':RUN_ID, 'out':str(OUT), 'datasets':{},
          'versions':{'python':platform.python_version(),'pydicom':pydicom.__version__},
          'scope':'Header and metadata audit; no pixel equivalence or clinical validation claims.'}
print('Output:', OUT)

## 1. Fungsi audit yang dapat dilanjutkan
Cache header disimpan setiap 250 berkas dan diberi identitas berdasarkan daftar isi arsip. Jika runtime putus, jalankan notebook lagi; header yang sudah berhasil dibaca digunakan kembali. Berkas yang gagal dibaca dicoba ulang. Nama, tanggal lahir, dan isi laporan medis pasien tidak diambil dari header. Tidak ada citra yang dihapus atau diberi label berdasarkan urutan berkas.

In [ ]:
def atomic_csv(table, path):
    path = Path(path)
    temp = path.with_suffix(path.suffix+'.partial')
    table.to_csv(temp,index=False)
    os.replace(temp,path)

def index_hash(z):
    items=[{'member':m.filename,'bytes':m.file_size,'compressed_bytes':m.compress_size,
            'crc32':f'{m.CRC:08x}'} for m in z.infolist() if not m.is_dir()]
    return hashlib.sha256(json.dumps(items,sort_keys=True).encode()).hexdigest()

def source_archive(dataset):
    source=ARCHIVES[dataset]
    if not source.is_file():raise FileNotFoundError(str(source))
    with zipfile.ZipFile(source) as z:
        fingerprint=index_hash(z)
        if len(z.namelist())!=len(set(z.namelist())):
            raise ValueError('Duplicate ZIP member names require manual inspection')
    if dataset in EXPECTED_INDEX and fingerprint!=EXPECTED_INDEX[dataset]:
        raise ValueError('Archive index differs from the uploaded B00 inventory')
    if not COPY_ARCHIVES_LOCAL:return source,fingerprint
    LOCAL_ARCHIVES.mkdir(parents=True,exist_ok=True)
    target=LOCAL_ARCHIVES/f'{dataset}_{fingerprint[:16]}.zip'
    if not target.exists() or target.stat().st_size!=source.stat().st_size:
        if shutil.disk_usage(LOCAL_ARCHIVES).free < source.stat().st_size + 1024**3:
            raise OSError('Insufficient runtime disk space for archive copy')
        print('Copying compressed archive:',source.name,flush=True)
        temp=target.with_suffix('.partial')
        shutil.copyfile(source,temp);os.replace(temp,target)
    with zipfile.ZipFile(target) as z:
        if index_hash(z)!=fingerprint:raise ValueError('Local archive index mismatch')
    return target,fingerprint

def scan_headers(dataset,path,fingerprint):
    cache=CACHE/f'{dataset}_{fingerprint}.csv'
    records={}
    if cache.is_file():
        old=pd.read_csv(cache,dtype=str,keep_default_na=False)
        records={r['member']:r for r in old.to_dict('records') if r.get('status')=='OK'}
    with zipfile.ZipFile(path) as z:
        members=[m for m in z.infolist() if m.filename.lower().endswith('.dcm')]
        valid_names={m.filename for m in members}
        records={k:v for k,v in records.items() if k in valid_names}
        tags=['PatientID','StudyInstanceUID','SeriesInstanceUID','SOPInstanceUID',
              'Laterality','ImageLaterality','PatientOrientation','ViewPosition',
              'ViewCodeSequence','Rows','Columns','BitsStored','PhotometricInterpretation',
              'Manufacturer','ManufacturerModelName','InstanceNumber','ImageType','SeriesDescription']
        for i,m in enumerate(members,1):
            if m.filename not in records:
                row={'member':m.filename,'bytes':m.file_size,'crc32':f'{m.CRC:08x}'}
                try:
                    with z.open(m) as f:ds=pydicom.dcmread(f,stop_before_pixels=True,specific_tags=tags)
                    row.update({tag:str(getattr(ds,tag,'')) for tag in tags if tag!='ViewCodeSequence'})
                    row['view_codes']=json.dumps([{'value':str(getattr(v,'CodeValue','')),
                        'meaning':str(getattr(v,'CodeMeaning',''))} for v in getattr(ds,'ViewCodeSequence',[])])
                    row['transfer_syntax']=str(getattr(ds.file_meta,'TransferSyntaxUID',''))
                    row['status']='OK'
                except Exception as exc:row.update(status='READ_ERROR',error=f'{type(exc).__name__}: {exc}')
                records[m.filename]=row
            if i%250==0:
                atomic_csv(pd.DataFrame(records.values()),cache)
                print(dataset,i,'/',len(members),flush=True)
    table=pd.DataFrame(records.values()).fillna('')
    atomic_csv(table,cache)
    atomic_csv(table,OUT/f'{dataset}_dicom_headers.csv')
    return table

def canonical_side(row):
    mapping={'L':'L','LEFT':'L','R':'R','RIGHT':'R'}
    values=[str(row.get(k,'')).strip().upper() for k in ['ImageLaterality','Laterality']]
    recognized={mapping[v] for v in values if v in mapping}
    if len(recognized)>1:return 'CONFLICT'
    return next(iter(recognized)) if recognized else ''

def header_summary(h):
    ok=h[h.status=='OK']
    uids=ok.get('SOPInstanceUID',pd.Series(dtype=str))
    nonempty=uids[uids!='']
    return {'headers':len(h),'read_errors':int((h.status!='OK').sum()),
            'empty_sop_uids':int((uids=='').sum()),'duplicate_sop_rows':int(nonempty.duplicated().sum()),
            'laterality':ok.get('Laterality',pd.Series(dtype=str)).value_counts().to_dict(),
            'image_laterality':ok.get('ImageLaterality',pd.Series(dtype=str)).value_counts().to_dict(),
            'view_position':ok.get('ViewPosition',pd.Series(dtype=str)).value_counts().to_dict(),
            'bits_stored':ok.get('BitsStored',pd.Series(dtype=str)).value_counts().to_dict()}

## 2. CMMD: label harus sesuai sisi payudara
Sheet klinis memiliki satu baris per catatan sisi/abnormalitas, bukan selalu satu baris per pasien. Empat citra tidak otomatis mendapat label yang sama. CMMD menghasilkan **candidate mapping** dari metadata klinis dan tag sisi eksplisit. Jika tag sisi kosong atau bertentangan, baris diberi status pending; tidak memakai dugaan nomor urut 1–2/3–4. Pemetaan ini masih perlu ditinjau sebelum split CMMD dibuat.

In [ ]:
try:
    path,fp=source_archive('cmmd')
    cmmd_headers=scan_headers('cmmd',path,fp)
    with zipfile.ZipFile(path) as z:
        clinical=pd.read_excel(io.BytesIO(z.read('CMMD_clinicaldata_revision.xlsx')),dtype=str).fillna('')
        archive_meta=pd.read_csv(io.BytesIO(z.read('metadata.csv')),dtype=str,keep_default_na=False)
    clinical.columns=clinical.columns.str.strip()
    clinical['patient_id']=clinical['ID1'].str.strip()
    clinical['side']=clinical['LeftRight'].str.strip().str.upper()
    clinical['label']=clinical['classification'].str.strip().str.casefold().map({'benign':0,'malignant':1})
    clinical['clinical_key']=clinical.patient_id+'|'+clinical.side
    atomic_csv(clinical,OUT/'cmmd_clinical.csv')
    atomic_csv(archive_meta,OUT/'cmmd_archive_metadata.csv')
    invalid=clinical.label.isna() | ~clinical.side.isin(['L','R']) | ~clinical.patient_id.str.match(r'^D[12]-\d+$')
    if invalid.any():raise ValueError(f'{int(invalid.sum())} invalid clinical label/ID/side rows; inspect exported CSV')
    grouped=clinical.groupby('clinical_key').agg(label=('label','first'),label_count=('label','nunique'),
                                               clinical_rows=('label','size'))
    candidates=cmmd_headers.copy()
    candidates['patient_from_path']=candidates.member.str.extract(r'(D[12]-\d+)',expand=False).fillna('')
    candidates['patient_from_header']=candidates.get('PatientID',pd.Series('',index=candidates.index)).str.extract(r'(D[12]-\d+)',expand=False).fillna('')
    candidates['side']=candidates.apply(canonical_side,axis=1)
    candidates['clinical_key']=candidates.patient_from_path+'|'+candidates.side
    candidates=candidates.merge(grouped,left_on='clinical_key',right_index=True,how='left',validate='many_to_one')
    candidates['mapping_status']='MAPPED_BY_EXPLICIT_SIDE'
    candidates.loc[candidates.label.isna(),'mapping_status']='NO_CLINICAL_SIDE_MATCH'
    candidates.loc[candidates.label_count.fillna(0)>1,'mapping_status']='CONFLICTING_CLINICAL_LABELS'
    candidates.loc[candidates.side=='','mapping_status']='MISSING_DICOM_SIDE'
    candidates.loc[candidates.side=='CONFLICT','mapping_status']='CONFLICTING_DICOM_SIDE'
    candidates.loc[candidates.patient_from_header=='','mapping_status']='MISSING_HEADER_PATIENT_ID'
    candidates.loc[(candidates.patient_from_header!='') & (candidates.patient_from_header!=candidates.patient_from_path),
                   'mapping_status']='PATIENT_ID_MISMATCH'
    candidates.loc[candidates.status!='OK','mapping_status']='HEADER_READ_ERROR'
    candidates.loc[candidates.mapping_status!='MAPPED_BY_EXPLICIT_SIDE','label']=np.nan
    atomic_csv(candidates,OUT/'cmmd_candidate_mapping.csv')
    summary=header_summary(cmmd_headers)
    summary.update(status='CANDIDATE_MAPPING_REQUIRES_REVIEW',index_sha256=fp,
                   clinical_rows=len(clinical),clinical_patients=clinical.patient_id.nunique(),
                   mapping_counts=candidates.mapping_status.value_counts().to_dict(),
                   mapped_label_counts=candidates.dropna(subset=['label']).label.value_counts().to_dict())
    report['datasets']['cmmd']=summary
    print(json.dumps(summary,indent=2,default=str))
except Exception as exc:
    report['datasets']['cmmd']={'status':'AUDIT_ERROR','error':f'{type(exc).__name__}: {exc}'}
    print(report['datasets']['cmmd'])

## 3. INbreast: CSV yang benar dan audit identitas pasien
CSV dibaca dengan titik koma. Field `Patient ID=removed` tidak dipakai sebagai identitas pasien. Token anonim pada nama file disimpan sebagai **candidate patient ID**, lalu dibandingkan dengan header DICOM. Tidak mencocokkan Sheet1 dan Sheet2 XLS berdasarkan urutan baris, dan tidak menyalin diagnosis bebas ke label secara otomatis.

Candidate cohort strict: BI-RADS 1/2 → 0; 5/6 → 1; kategori lain dikeluarkan. Ini label operasional berbasis BI-RADS, bukan klaim bahwa semua positif telah terbukti melalui patologi. Identitas pasien dan konsistensi label per sisi tetap ditinjau sebelum split.

In [ ]:
try:
    path,fp=source_archive('inbreast')
    in_headers=scan_headers('inbreast',path,fp)
    with zipfile.ZipFile(path) as z:
        names=[n for n in z.namelist() if Path(n).name.casefold()=='inbreast.csv']
        if len(names)!=1:raise ValueError('Expected exactly one INbreast.csv')
        clinical=pd.read_csv(io.BytesIO(z.read(names[0])),sep=';',dtype=str,keep_default_na=False)
    clinical.columns=clinical.columns.str.strip()
    required={'File Name','Laterality','View','Bi-Rads','Acquisition date'}
    if not required.issubset(clinical):raise ValueError(f'Missing columns: {required-set(clinical)}')
    clinical['image_id']=clinical['File Name'].str.strip()
    if clinical.image_id.duplicated().any():raise ValueError('Duplicate image IDs in INbreast.csv')
    clinical['birads_normalized']=clinical['Bi-Rads'].str.strip().str.casefold()
    clinical['label_candidate']=clinical.birads_normalized.map({'1':0,'2':0,'5':1,'6':1})
    atomic_csv(clinical,OUT/'inbreast_clinical.csv')
    tokens=in_headers.member.map(lambda s:Path(s).name).str.extract(r'^(\d+)_([0-9a-fA-F]+)_MG_([RL])_([^_]+)_ANON\.dcm$')
    tokens.columns=['image_id','patient_token','filename_side','filename_view']
    candidates=pd.concat([in_headers,tokens],axis=1)
    candidates['dicom_side']=candidates.apply(canonical_side,axis=1)
    if tokens.isna().any().any():raise ValueError('Unexpected INbreast DICOM filename; inspect headers export')
    candidates=candidates.merge(clinical,on='image_id',how='left',validate='one_to_one',indicator=True)
    if not candidates['_merge'].eq('both').all() or len(candidates)!=len(clinical):
        raise ValueError('INbreast image IDs are not a complete one-to-one match')
    candidates['side_disagreement']=(candidates.filename_side!=candidates.Laterality_y if 'Laterality_y' in candidates
                                     else candidates.filename_side!=candidates['Laterality'])
    # The clinical Laterality column is suffixed if DICOM Laterality also exists.
    clinical_side='Laterality_y' if 'Laterality_y' in candidates else 'Laterality'
    candidates['breast_visit_key']=candidates.patient_token+'|'+candidates[clinical_side]+'|'+candidates['Acquisition date']
    atomic_csv(candidates,OUT/'inbreast_candidate_mapping.csv')
    pairs=candidates[['patient_token','PatientID']].drop_duplicates()
    summary=header_summary(in_headers)
    summary.update(status='CANDIDATE_MAPPING_REQUIRES_REVIEW',index_sha256=fp,
       csv_rows=len(clinical),filename_patient_tokens=candidates.patient_token.nunique(),
       header_patient_ids=candidates.PatientID.nunique(),
       tokens_with_multiple_header_ids=int((pairs.groupby('patient_token').PatientID.nunique()>1).sum()),
       header_ids_with_multiple_tokens=int((pairs.groupby('PatientID').patient_token.nunique()>1).sum()),
       filename_clinical_side_disagreements=int(candidates.side_disagreement.sum()),
       birads_counts=clinical.birads_normalized.value_counts().to_dict(),
       strict_candidate_images=int(candidates.label_candidate.notna().sum()),
       strict_candidate_label_counts=candidates.label_candidate.value_counts().to_dict(),
       breast_visits_with_mixed_binary_labels=int((candidates.groupby('breast_visit_key').label_candidate.nunique()>1).sum()))
    report['datasets']['inbreast']=summary
    print(json.dumps(summary,indent=2,default=str))
except Exception as exc:
    report['datasets']['inbreast']={'status':'AUDIT_ERROR','error':f'{type(exc).__name__}: {exc}'}
    print(report['datasets']['inbreast'])

## 4. MIAS versi penuh: baca PDF asli dan seluruh header PGM
Versi ini berisi 322 PGM dengan dimensi bervariasi; jangan memakai asumsi mini-MIAS 1024×1024. Notebook mengekstrak teks **dan menyalin PDF README utuh** agar tabel label/pasangan pasien bisa diperiksa dari layout aslinya. Label tidak diambil dari sufiks nama berkas atau nomor ganjil/genap tanpa memeriksa dokumentasi.

In [ ]:
try:
    from PIL import Image
    path,fp=source_archive('mias')
    rows=[];pdfs=[]
    with zipfile.ZipFile(path) as z:
        for m in z.infolist():
            if m.filename.lower().endswith('.pgm'):
                with z.open(m) as f:
                    im=Image.open(f)
                    rows.append({'member':m.filename,'width':im.width,'height':im.height,'mode':im.mode,
                                 'bytes':m.file_size,'crc32':f'{m.CRC:08x}'})
            elif m.filename.lower().endswith('.pdf'):
                data=z.read(m)
                reader=PdfReader(io.BytesIO(data))
                text='\n\n'.join(f'PAGE {i+1}\n'+(page.extract_text() or '') for i,page in enumerate(reader.pages))
                safe=Path(m.filename).name.replace(' ','_')
                (OUT/('mias_'+safe)).write_bytes(data)
                (OUT/('mias_'+Path(safe).stem+'.txt')).write_text(text,encoding='utf-8')
                pdfs.append({'member':m.filename,'pages':len(reader.pages),'extracted_characters':len(text)})
                print('PDF:',m.filename,'Pages:',len(reader.pages))
                if 'readme' in safe.casefold():print(text[:6000])
    table=pd.DataFrame(rows)
    atomic_csv(table,OUT/'mias_pgm_headers.csv')
    report['datasets']['mias']={'status':'PDF_LABEL_MAPPING_PENDING_REVIEW','index_sha256':fp,
                               'images':len(table),'dimension_pairs':len(table[['width','height']].drop_duplicates()),
                               'pdfs':pdfs}
    display(table.head())
except Exception as exc:
    report['datasets']['mias']={'status':'AUDIT_ERROR','error':f'{type(exc).__name__}: {exc}'}
    print(report['datasets']['mias'])

## 5. CBIS-DDSM: buat dan bekukan split pasien baru
Pembagian 70/15/15 dibuat dari tabel pasien, distratifikasi berdasarkan apakah pasien memiliki sedikitnya satu citra positif. Rasio citra tidak harus tepat 70/15/15. Seluruh model harus memakai split yang sama. Jangan gunakan checkpoint penelitian lama untuk benchmark split baru.

Jika CSV memiliki hash berbeda dari B00, proses CBIS berhenti untuk diperiksa; jangan menghapus pemeriksaan hash hanya agar berjalan. Split disimpan pada folder baru. Manifest berisi path sumber, belum membuktikan semua berkas citra dapat dibuka. Lokasi JPEG diperiksa terpisah di bawahnya.

In [ ]:
def build_cbis_splits(table, seed):
    table=table.copy()
    required={'image_path','patient_id','cancer'}
    if not required.issubset(table):raise ValueError('Missing required CBIS columns')
    if table[list(required)].isna().any().any():raise ValueError('Null CBIS path/patient/label')
    for key in ['image_path','patient_id']:
        if table[key].astype(str).str.strip().eq('').any():raise ValueError('Empty path or patient')
    table['cancer']=pd.to_numeric(table.cancer,errors='raise')
    if not table.cancer.isin([0,1]).all():raise ValueError('Non-binary labels')
    if table.image_path.duplicated().any():raise ValueError('Duplicate paths')
    table['cancer']=table.cancer.astype(int)
    patients=table.groupby('patient_id',sort=True).cancer.max().reset_index()
    n_test=int(np.ceil(len(patients)*0.15))
    n_val=int(np.ceil(len(patients)*0.15))
    remaining,test=train_test_split(patients,test_size=n_test,random_state=seed,stratify=patients.cancer)
    train,val=train_test_split(remaining,test_size=n_val,random_state=seed,stratify=remaining.cancer)
    allocation={pid:split for split,frame in [('train',train),('validation',val),('test',test)] for pid in frame.patient_id}
    table['split']=table.patient_id.map(allocation)
    table['label']=table.cancer
    table['dataset']='cbis_ddsm'
    table['sample_id']=table.image_path.map(lambda s:hashlib.sha256(str(s).encode()).hexdigest())
    if table.groupby('patient_id').split.nunique().max()!=1:raise AssertionError('Patient overlap')
    for split,frame in table.groupby('split'):
        if set(frame.label)!={0,1}:raise ValueError(f'{split} lacks a class')
    return table.sort_values('sample_id').reset_index(drop=True)

try:
    digest=hashlib.sha256(CBIS_CSV.read_bytes()).hexdigest()
    if digest!=EXPECTED_CBIS_SHA256:raise ValueError('cbis_all.csv differs from the B00 inventory SHA256')
    cbis=build_cbis_splits(pd.read_csv(CBIS_CSV,dtype={'patient_id':str}),SPLIT_SEED)
    manifest_dir=BASE/'manifests'/f'cbis_ddsm_patient701515_seed{SPLIT_SEED}_{digest[:12]}'
    manifest_dir.mkdir(parents=True,exist_ok=True)
    for split,frame in cbis.groupby('split'):
        target=manifest_dir/f'{split}.csv'
        payload=frame.to_csv(index=False).encode()
        if target.exists() and target.read_bytes()!=payload:
            raise ValueError(f'Existing frozen split differs: {target}')
        if not target.exists():target.write_bytes(payload)
        (OUT/f'cbis_{split}.csv').write_bytes(payload)
    summary=[]
    for split,frame in cbis.groupby('split'):
        pg=frame.groupby('patient_id').label.max()
        summary.append({'split':split,'images':len(frame),'patients':len(pg),
                        'positive_images':int(frame.label.sum()),'negative_images':int((frame.label==0).sum()),
                        'patients_with_positive_image':int(pg.sum())})
    status={'status':'PATIENT_SPLITS_FROZEN_IMAGE_ACCESS_PENDING','manifest_dir':str(manifest_dir),
            'source_sha256':digest,'split_seed':SPLIT_SEED,'summary':summary,
            'pixel_duplicates_checked':False,'historical_data_previously_used':True}
    (manifest_dir/'split_config.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
    report['datasets']['cbis']=status
    display(pd.DataFrame(summary))
except Exception as exc:
    report['datasets']['cbis']={'status':'SPLIT_ERROR','error':f'{type(exc).__name__}: {exc}'}
    print(report['datasets']['cbis'])

## 6. Cari lokasi JPEG CBIS untuk notebook training
Pencarian terbatas pada kandidat folder yang dikonfigurasi dan ZIP bernama mengandung `cbis` pada folder datasets. ZIP hanya dibaca daftar isinya. Jika ada beberapa lokasi, notebook mencatat semua kandidat untuk ditentukan pada notebook training; tidak mencampur salinan. Semua pemeriksaan ini masih berbasis keberadaan path/anggota ZIP, bukan kesetaraan piksel.

In [ ]:
if report['datasets'].get('cbis',{}).get('status','').startswith('PATIENT_SPLITS'):
    def jpeg_key(s):
        s=str(s).replace('\\','/')
        return s.split('/jpeg/',1)[-1] if '/jpeg/' in s else s
    keys=cbis.image_path.map(jpeg_key)
    locations=[]
    for root in CBIS_IMAGE_ROOTS:
        if not root.exists():continue
        trials=[root,root/'cbis_ddsm_awsaf',root/'CBIS-DDSM']
        for base in trials:
            for style in ['relative','jpeg']:
                sample=(base/str(cbis.image_path.iloc[0]) if style=='relative' else base/'jpeg'/keys.iloc[0])
                if not sample.is_file():continue
                count=sum((base/str(s) if style=='relative' else base/'jpeg'/k).is_file()
                          for s,k in zip(cbis.image_path,keys))
                locations.append({'type':'directory','base':str(base),'path_style':style,'matched_images':count})
    zips=[Path(CBIS_ZIP_OVERRIDE)] if CBIS_ZIP_OVERRIDE.strip() else [
        p for root in [PROJECT/'datasets',Path('/content/drive/MyDrive/datasets')] if root.is_dir()
        for p in root.glob('*.zip') if 'cbis' in p.name.casefold()]
    for p in zips:
        try:
            with zipfile.ZipFile(p) as z:
                names=[n for n in z.namelist() if n.lower().endswith(('.jpg','.jpeg'))]
                mapped=Counter(jpeg_key(n) for n in names)
            count=sum(mapped.get(k,0)==1 for k in keys)
            locations.append({'type':'zip','path':str(p),'matched_images':count,
                              'ambiguous_keys':sum(mapped.get(k,0)>1 for k in keys)})
        except Exception as exc:locations.append({'type':'zip','path':str(p),'error':str(exc)})
    report['datasets']['cbis']['image_locations']=locations
    report['datasets']['cbis']['full_image_locator_found']=any(x.get('matched_images')==len(cbis) for x in locations)
    if report['datasets']['cbis']['full_image_locator_found']:
        report['datasets']['cbis']['status']='PATIENT_SPLITS_FROZEN_IMAGE_LOCATOR_FOUND'
    print(json.dumps(locations,indent=2))

## 7. Simpan dan unduh hasil
Unggah `B01_preparation_report.zip`. Isinya mencakup header dan metadata lengkap, PDF MIAS, candidate mapping, serta split CBIS. Tanpa piksel DICOM/PGM/JPEG. Notebook tidak menyatakan CMMD/INbreast/MIAS siap training hanya karena pembacaan metadata berhasil.

In [ ]:
report_path=OUT/'B01_report.json'
report_path.write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
bundle=OUT/'B01_preparation_report.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file() and path!=bundle:z.write(path,arcname=path.name)
display(pd.DataFrame([{'dataset':k,'status':v['status'],'error':v.get('error','')} for k,v in report['datasets'].items()]))
print('Saved:',bundle)
print('Report only; no training has run.')
from google.colab import files
files.download(str(bundle))